In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import NoSuchElementException
from bs4 import BeautifulSoup
import time
import re
import pandas as pd
from datetime import date

## Crawling!

#### 1 : main title -> click
#### 2 : subtitle -> click
#### 3 : content -> get "Quesion" and "Main contents"
#### 4 : English content -> click
#### 5 : English content -> get "Quesion" and "Main contents"

In [ ]:
driver = webdriver.Chrome()
driver.get("https://www.gotquestions.org/Korean/")

a_tags=driver.find_elements(By.TAG_NAME,"a")
print(len(a_tags))

In [ ]:
# 영어를 포함한 값 추출해보기 - test
final=pd.DataFrame(columns=["crawling_date","big_title_kor","Question_KOR","Answer_KOR","URL_KOR","Question_ENG","Answer_ENG","URL_ENG"])

In [ ]:
for i in range(1, 52): # 첫 번째 클릭 (대주제)
    

    # 1 : main title -> click
    try :
        element = driver.find_element(By.XPATH, f'/html/body/main/section[1]/div/strong/a[{i}]') # 버튼의 xpath를 이용 (1, 2, 3, ...)
        big_title_kor = driver.find_element(By.XPATH, f'/html/body/main/section[1]/div/strong/a[{i}]').text # 대주제도 데이터 프레임에 넣기
        driver.execute_script("arguments[0].click();", element)
        time.sleep(0.5)
    

    except NoSuchElementException:
        time.sleep(0.5)
        break

    # 2 : subtitle -> click
    for ii in range(1, 300): # 두 번째 클릭 (소주제) 

        # 소주제 개수는 대주제마다 달라서, range()로 일단 1부터 120까지 범위 정해놓고
        # 반복문을 실행할 수 없는 상황, 찾으려는 요소가 없어서 더 이상 작업할 수 없을 때 반복문 종료
        try:
            element = driver.find_element(By.XPATH, f'/html/body/main/section[1]/div/strong/a[{ii}]') # 버튼의 xpath를 이용 (1, 2, 3, ...)
            driver.execute_script("arguments[0].click();", element)
            time.sleep(0.5)
        
        # 만약 버튼이 없다면 초기 화면으로 돌아간 후 초기 반복문으로 이동
        except NoSuchElementException:
            driver.get("https://www.gotquestions.org/Korean/")
            time.sleep(0.5)
            break
        
        # 3 : content -> get "Quesion" and "Main contents"
        html = driver.page_source
        soup = BeautifulSoup(html, 'html.parser')
            
        # crawling 하는 날짜 date 넣기
        crawling_date = date.today()
        # 제목(소주제)는 <h1></h1> 사이에 존재
        title_kor = soup.find('h1').get_text(strip = True)
        # 현재 url
        url_kor = driver.current_url
        
        # 답변 내용은 "답변"과 "English" 사이에 존재한다는 공통점
        content_str_kor = str(soup.get_text(separator="\n"))
        start_kor = content_str_kor.find('답변') + len('답변') # 답변의 시작 지점
        end_kor = content_str_kor.find('English') # 답변의 끝 지점
        answer_html_kor = content_str_kor[start_kor : end_kor]
        answer_kor = BeautifulSoup(answer_html_kor, 'html.parser').get_text("\n", strip=True) # <>와 같은 태그 제거하여 텍스트만 남기기

        # ----------- #

        # 4 : English content -> click
        element = driver.find_element(By.LINK_TEXT, "English")
        driver.execute_script("arguments[0].click();", element)
        time.sleep(0.5)

        html = driver.page_source
        soup = BeautifulSoup(html, 'html.parser')
            
        # 5 : English content -> get "Quesion" and "Main contents"
        # 영어 제목(소주제) <h1></h1> 사이에 존재
        title_eng = soup.find('h1').get_text(strip = True)

        # 현재 url
        url_eng = driver.current_url
            
        # 답변 내용은 "Question"과 "Return to:" 사이에 존재한다는 공통점
        content_str_eng = str(soup.get_text(separator="\n"))
        start_eng = content_str_eng.find('Answer') + len('Answer') # 답변의 시작 지점
        end_eng = content_str_eng.find('Return to:') # 답변의 끝 지점
        answer_html_eng = content_str_eng[start_eng : end_eng]
        answer_eng = BeautifulSoup(answer_html_eng, 'html.parser').get_text("\n", strip=True) # <>와 같은 태그 제거하여 텍스트만 남기기

        
        # 파일 저장
        new_row = pd.DataFrame({"crawling_date":[crawling_date], "big_title_kor" : [big_title_kor],
                                "Question_KOR": [title_kor], "Answer_KOR": [answer_kor], "URL_KOR": [url_kor],
                                "Question_ENG": [title_eng], "Answer_ENG": [answer_eng], "URL_ENG": [url_eng]})
        final = pd.concat([final, new_row], ignore_index=True)
        
        driver.back()
        time.sleep(0.5)

        driver.back()
        time.sleep(0.5)

In [ ]:
final

In [ ]:
# 추후 자료 매칭을 위해
final["URL_KOR_dup"]=final["URL_KOR"].str.replace(r"https://www.gotquestions.org/Korean/Korean-", "", regex=True)
final["URL_KOR_dup"]=final["URL_KOR_dup"].str.replace(r".html", "", regex=True)

In [ ]:
final.to_excel(f"/Users/haley/Desktop/2025-1/DS1/DataScience_Capstone/need_preprocessing/GotQuestions/GotQuestions_raw_Kor_{crawling_date}.xlsx",index=False)

In [ ]:
stop!

## Preprocessing

#### duplicated check

In [ ]:
import pandas as pd
import re

In [ ]:
final=pd.read_excel("/Users/haley/Desktop/2025-1/DS1/DataScience_Capstone/need_preprocessing/GotQuestions/GotQuestions_raw_Kor_2025-06-16.xlsx")

In [ ]:
# url 주소가 duplicated 되는 것?
final["URL_KOR_dup"]
len(final)

In [ ]:
# duplicated row
print(final[final.duplicated("URL_KOR_dup")].shape[0])

In [ ]:
final[final.duplicated("URL_KOR_dup")].sort_values("URL_KOR_dup").head()

In [ ]:
# duplicated row remove
final_2=final.drop_duplicates(subset="URL_KOR_dup")
len(final_2)

#### ENG Answer Preprocessing 

In [ ]:
# 1
# Find : "Have you ~ button below ~""
final_2["Answer_ENG"].apply(lambda x: re.findall(r"\nHave you[\s\S]*?button below[\s\S]+",x))

In [ ]:
# Remove : "Have you ~ button below ~""
final_2["Answer_ENG"] = final_2["Answer_ENG"].str.replace(r"\nHave you[\s\S]*?button below[\s\S]+", "", regex=True)

In [ ]:
final_2["Answer_ENG"]

In [ ]:
# 2
# Find : "Related Articles ~"
final_2["Answer_ENG"].apply(lambda x: re.findall(r"\nRelated Articles[\s\S]+",x))

In [ ]:
# Remove : "Related Articles ~"
final_2["Answer_ENG"] = final_2["Answer_ENG"].str.replace(r"\nRelated Articles[\s\S]+", "", regex=True)

In [ ]:
final_2["Answer_ENG"]

In [ ]:
# 3
# Find : "For Further Study ~"
final_2["Answer_ENG"].apply(lambda x: re.findall(r"\nFor Further Study[\s\S]+",x))

In [ ]:
# Remove : "For Further Study ~"
final_2["Answer_ENG"] = final_2["Answer_ENG"].str.replace(r"\nFor Further Study[\s\S]+", "", regex=True)

In [ ]:
final_2["Answer_ENG"]

In [ ]:
# 4
# Find : "\n{1,}"
final_2["Answer_ENG"].apply(lambda x: re.findall(r"\n{1,}",x))[150]

In [ ]:
# Remove : "\n{1,}"
final_2["Answer_ENG"] = final_2["Answer_ENG"].str.replace(r"\n{1,}", "", regex=True)

In [ ]:
final_2["Answer_ENG"][121]

In [ ]:
# 5
#...? anything else..?

#### Korean Anwer Preprocessing

In [ ]:
final_2["Answer_KOR"][1]

In [ ]:
# 1
# Find : "이곳의 글을 읽고 ~"
final_2["Answer_KOR"].apply(lambda x: re.findall(r"\n이곳의 글을 읽고[\s\S]+",x))

In [ ]:
# Remove : "이곳의 글을 읽고 ~"
final_2["Answer_KOR"]=final_2["Answer_KOR"].str.replace(r"\n이곳의 글을 읽고[\s\S]+", "", regex=True)

In [ ]:
final_2["Answer_KOR"]

In [ ]:
# 2
# Find : "\n{1,}"
final_2["Answer_KOR"].apply(lambda x: re.findall(r"\n{1,}",x))[172]

In [ ]:
# Remove : "\n{1,}"
final_2["Answer_KOR"] = final_2["Answer_KOR"].str.replace(r"\n{1,}", "", regex=True)

In [ ]:
final_2["Answer_KOR"][177]

In [ ]:
final.to_excel(f"/Users/haley/Desktop/2025-1/DS1/DataScience_Capstone/need_preprocessing/GotQuestions/GotQuestions_raw_Kor_2025-06-16.xlsx",index=False)

### Merge

In [ ]:
old=pd.read_excel("/Users/haley/Desktop/2025-1/DS1/DataScience_Capstone/need_preprocessing/GotQuestions/GotQuestions_raw_Kor_2025-04-29.xlsx")
old

In [ ]:
# 자료 매칭을 위해
old["URL_KOR_dup"]=old["URL_KOR"].str.replace(r"https://www.gotquestions.org/Korean/Korean-", "", regex=True)
old["URL_KOR_dup"]=old["URL_KOR_dup"].str.replace(r".html", "", regex=True)

old.head()
## 추후 이 셀 제거

In [ ]:
len(old)

In [ ]:
# duplicated row remove
old=old.drop_duplicates(subset="URL_KOR_dup")
len(old)

## 추후 이 셀 제거

In [ ]:
compare = pd.merge(final_2, old, on = 'URL_KOR_dup', how = 'left')
compare

In [ ]:
compare[compare['crawling_date_y'].isna()]

In [ ]:
#final.to_excel(f"/Users/haley/Desktop/2025-1/DS1/DataScience_Capstone/need_preprocessing/GotQuestions/GotQuestions_KOR_final.xlsx",index=False)